# ✉️ Messages (Cloudflare Workers AI)
  <img src="./assets/LC_Messages.png" width="500">

Same lesson as `L2_messages.ipynb`, but the chat model is **`ChatCloudflareWorkersAI`** (Workers AI is not a simple `"provider:model"` string like OpenAI here). Set `CF_AI_API_TOKEN` and `CF_ACCOUNT_ID` in `.env` (see `example.env`); optionally override the model with `CF_WORKERS_AI_MODEL`. Requires `langchain-cloudflare` (see `pyproject.toml`).

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

## Setup

Load and/or check environment variables. For this notebook, ensure **`CF_AI_API_TOKEN`** and **`CF_ACCOUNT_ID`** are set (and optionally **`CF_WORKERS_AI_MODEL`**). Keys listed in `example.env` are checked below.

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env, doublecheck_pkgs

load_dotenv()

doublecheck_env("example.env")
doublecheck_pkgs(pyproject_path="pyproject.toml", verbose=True)

OPENAI_API_KEY=****here
LANGSMITH_API_KEY=****3e8d
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials
GROQ_API_KEY=****MPIf
CF_AI_API_TOKEN=****bf9b
CF_ACCOUNT_ID=****1f1e
Python 3.11.11 satisfies requires-python: >=3.11,<3.14
package                | required | installed | status              | path                                                                
---------------------- | -------- | --------- | ------------------- | --------------------------------------------------------------------
langgraph              | >=1.0.0  | 1.0.10rc1 | ⚠️ Version mismatch | C:\MarkDev\lca-langchainV1-essentials\python\.venv\Lib\site-packages
langchain              | >=1.0.0  | 1.2.6     | ✅ OK                | C:\MarkDev\lca-langchainV1-essentials\python\.venv\Lib\site-packages
langchain-core         | >=1.0.0  | 1.2.22    | ✅ OK                | C:\MarkDev\lca-langchainV1-essentials\python\.venv\Lib\site-packages
langchain-openai       | >=1.0.0  | 1.1.7     | ✅ OK                | C:\MarkDe

## Human👨‍💻 and AI 🤖 Messages

In [2]:
import os

from langchain.agents import create_agent
from langchain_cloudflare.chat_models import ChatCloudflareWorkersAI
from langchain_core.messages import HumanMessage

lc_model = ChatCloudflareWorkersAI(
    model=os.getenv("CF_WORKERS_AI_MODEL", "@cf/moonshotai/kimi-k2.5"),
    temperature=0,
    max_tokens=1024,
)

agent = create_agent(
    model=lc_model,
    system_prompt="You are a full-stack comedian"
)

In [3]:
human_msg = HumanMessage("Hello, how are you?")

result = agent.invoke({"messages": [human_msg]})

In [4]:
print(result["messages"][-1].content)

[{'type': 'thinking', 'thinking': 'Okay, the user just said "Hello, how are you?" in a very casual, friendly way. Hmm, this seems like a standard greeting to start a conversation—probably testing if I\'m responsive or just being polite. \n\nI should keep it warm and open-ended since they didn\'t give any specific context. No need to overcomplicate it; they might just want to chat or see how I handle small talk. \n\nNoting they used "Hello" with a comma—casual but not sloppy—so I\'ll match that tone: friendly but not overly familiar. Gotta avoid sounding robotic though. \n\n...Also, no signs of frustration or urgency in their message, so likely just a low-stakes check-in. Safe to assume they\'re in a neutral/good mood. \n\n*Prepares to reply with:* \n- Acknowledgment of their greeting \n- Light self-update (keeping it positive but genuine—no fake enthusiasm) \n- Turning it back to them to keep the convo flowing \n- Added a tiny emoji for approachability since text can feel cold \n\n*Dou

In [5]:
print(type(result["messages"][-1]))

<class 'langchain_core.messages.ai.AIMessage'>


In [6]:
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

human: Hello, how are you?

ai: [{'type': 'thinking', 'thinking': 'Okay, the user just said "Hello, how are you?" in a very casual, friendly way. Hmm, this seems like a standard greeting to start a conversation—probably testing if I\'m responsive or just being polite. \n\nI should keep it warm and open-ended since they didn\'t give any specific context. No need to overcomplicate it; they might just want to chat or see how I handle small talk. \n\nNoting they used "Hello" with a comma—casual but not sloppy—so I\'ll match that tone: friendly but not overly familiar. Gotta avoid sounding robotic though. \n\n...Also, no signs of frustration or urgency in their message, so likely just a low-stakes check-in. Safe to assume they\'re in a neutral/good mood. \n\n*Prepares to reply with:* \n- Acknowledgment of their greeting \n- Light self-update (keeping it positive but genuine—no fake enthusiasm) \n- Turning it back to them to keep the convo flowing \n- Added a tiny emoji for approachability s

### Altenative formats
#### Strings
There are situations where LangChain can infer the role from the context, and a simple string is enough to create a message. 

In [7]:
agent = create_agent(
    model=lc_model,
    system_prompt="You are a terse sports poet.",  # This is a SystemMessage under the hood
)

In [8]:
result = agent.invoke({"messages": "Tell me about baseball"})   # This is a HumanMessage under the hood
print(result["messages"][-1].content)

[{'type': 'thinking', 'thinking': 'Okay, the user asked me to "tell me about baseball" with a specific instruction to be a "terse sports poet." That\'s an interesting constraint—they want brevity and poetic flair, not a dry encyclopedia entry. \n\nHmm, judging by the phrasing, they might be testing if I can follow creative constraints while staying accurate. Or maybe they\'re tired of lengthy explanations and want something punchy that sticks in the mind. The "terse" demand suggests they value economy of words—every line must pull weight. \n\nI should avoid clichés like "crack of the bat" unless I twist them freshly. Focus on sensory details: the smell of grass, the tension in a pitcher\'s grip, the way time distorts during a 3-2 count. Baseball\'s poetry lives in its pauses—the dugout spit, the catcher\'s signals, the sudden explosion of action. \n\nAlso, must resist the urge to lecture about innings or ERAs. This isn\'t a tutorial; it\'s a haiku for the soul of the game. Short lines.

#### Dictionaries

In [9]:
result = agent.invoke(
    {"messages": {"role": "user", "content": "Write a haiku about sprinters"}}
)
print(result["messages"][-1].content)

[{'type': 'thinking', 'thinking': 'We need to produce a haiku (5-7-5 syllable) about sprinters. Should be terse sports poet style. Provide haiku. Ensure correct syllable count. Let\'s craft: "Lightning on the track" (5? Lightning(2) on(1) the(1) track(1) =5). Next line 7 syllables: "Muscles coil, then burst — ". Count: Muscles(2) coil(1), then(1) burst(1) — maybe dash counts? Let\'s count: Muscles(2) coil(1) =3, then(1)=4, burst(1)=5. Need 7. Add "like" and "fire". "Muscles coil, then burst like fire". Count: Muscles(2) coil(1)=3, then(1)=4, burst(1)=5, like(1)=6, fire(2)=8. Too many. Let\'s try different.\n\nLine2: "Feet kiss the earth, then fly". Count: Feet(1) kiss(1)=2, the(1)=3, earth(1)=4, then(1)=5, fly(1)=6. Need 7. Add "swift". "Feet kiss the earth, then fly swift". Count: Feet1 kiss1=2 the1=3 earth1=4 then1=5 fly1=6 swift1=7. Good.\n\nLine3 5 syllables: "Hearts race the wind". Count: Hearts1 race1=2 the1=3 wind1=4. Need 5. Add "fast". "Hearts race the wind fast". Count: Heart

There are multiple roles:
```python
messages = [
    {"role": "system", "content": "You are a sports poetry expert who completes haikus that have been started"},
    {"role": "user", "content": "Write a haiku about sprinters"},
    {"role": "assistant", "content": "Feet don't fail me..."}
]
```

## Output Format
### messages
Let's create a tool so agent will create some tool messages. 

In [10]:
from langchain_core.tools import tool

@tool
def check_haiku_lines(text: str):
    """Check if the given haiku text has exactly 3 lines.

    Returns None if it's correct, otherwise an error message.
    """
    # Split the text into lines, ignoring leading/trailing spaces
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    print(f"checking haiku, it has {len(lines)} lines:\n {text}")

    if len(lines) != 3:
        return f"Incorrect! This haiku has {len(lines)} lines. A haiku must have exactly 3 lines."
    return "Correct, this haiku has 3 lines."

In [11]:
agent = create_agent(
    model=lc_model,
    tools=[check_haiku_lines],
    system_prompt="You are a sports poet who only writes Haiku. You always check your work.",
)

In [12]:
result = agent.invoke({"messages": "Please write me a poem"})

In [15]:
result["messages"][-1].content

[{'type': 'thinking',
  'thinking': 'The user wants a poem. I am a sports poet who only writes Haiku. So I need to write a haiku about sports. I must check my work using the check_haiku_lines tool to ensure it has exactly 3 lines. I should first compose a haiku, then call the tool to verify. If it\'s correct, I can output it. If not, I need to adjust.\n\nLet\'s think of a sports haiku. For example: "Baseball diamond gleams / Bat cracks, ball soars over fence / Crowd erupts in cheer". Count syllables: Line1: Base-ball dia-mond gleams (5? Base-ball (2), dia-mond (3), gleams (1) = 6? Actually "Baseball" is 2 syllables (base-ball), "diamond" is 3 (dia-mond), "gleams" is 1, total 6. Need 5. Let\'s adjust.\n\nBetter: "Bat cracks in the sun / Ball flies over green outfield / Fans shout, \'Home run!\'" Count: Line1: Bat (1) cracks (1) in (1) the (1) sun (1) =5? Actually "Bat cracks in the sun": Bat(1) cracks(1) in(1) the(1) sun(1) =5. Good. Line2: Ball(1) flies(1) o-ver(2) green(1) out-field(2

In [16]:
print(len(result["messages"]))

2


In [17]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

Please write me a poem
================================== Ai Message ==================================

[{'type': 'thinking', 'thinking': 'The user wants a poem. I am a sports poet who only writes Haiku. So I need to write a haiku about sports. I must check my work using the check_haiku_lines tool to ensure it has exactly 3 lines. I should first compose a haiku, then call the tool to verify. If it\'s correct, I can output it. If not, I need to adjust.\n\nLet\'s think of a sports haiku. For example: "Baseball diamond gleams / Bat cracks, ball soars over fence / Crowd erupts in cheer". Count syllables: Line1: Base-ball dia-mond gleams (5? Base-ball (2), dia-mond (3), gleams (1) = 6? Actually "Baseball" is 2 syllables (base-ball), "diamond" is 3 (dia-mond), "gleams" is 1, total 6. Need 5. Let\'s adjust.\n\nBetter: "Bat cracks in the sun / Ball flies over green outfield / Fans shout, \'Home run!\'" Count: Li

### Other useful information
Above, the print messages have just been selecting pieces of the information stored in the messages list. Let's dig into all the information that is available!

In [18]:
result

{'messages': [HumanMessage(content='Please write me a poem', additional_kwargs={}, response_metadata={}, id='9d6f245c-a5a0-4874-89df-b38906f449ad'),
  AIMessage(content=[{'type': 'thinking', 'thinking': 'The user wants a poem. I am a sports poet who only writes Haiku. So I need to write a haiku about sports. I must check my work using the check_haiku_lines tool to ensure it has exactly 3 lines. I should first compose a haiku, then call the tool to verify. If it\'s correct, I can output it. If not, I need to adjust.\n\nLet\'s think of a sports haiku. For example: "Baseball diamond gleams / Bat cracks, ball soars over fence / Crowd erupts in cheer". Count syllables: Line1: Base-ball dia-mond gleams (5? Base-ball (2), dia-mond (3), gleams (1) = 6? Actually "Baseball" is 2 syllables (base-ball), "diamond" is 3 (dia-mond), "gleams" is 1, total 6. Need 5. Let\'s adjust.\n\nBetter: "Bat cracks in the sun / Ball flies over green outfield / Fans shout, \'Home run!\'" Count: Line1: Bat (1) crack

You can select just the last message, and you can see where the final message is coming from.

In [19]:
result["messages"][-1]

AIMessage(content=[{'type': 'thinking', 'thinking': 'The user wants a poem. I am a sports poet who only writes Haiku. So I need to write a haiku about sports. I must check my work using the check_haiku_lines tool to ensure it has exactly 3 lines. I should first compose a haiku, then call the tool to verify. If it\'s correct, I can output it. If not, I need to adjust.\n\nLet\'s think of a sports haiku. For example: "Baseball diamond gleams / Bat cracks, ball soars over fence / Crowd erupts in cheer". Count syllables: Line1: Base-ball dia-mond gleams (5? Base-ball (2), dia-mond (3), gleams (1) = 6? Actually "Baseball" is 2 syllables (base-ball), "diamond" is 3 (dia-mond), "gleams" is 1, total 6. Need 5. Let\'s adjust.\n\nBetter: "Bat cracks in the sun / Ball flies over green outfield / Fans shout, \'Home run!\'" Count: Line1: Bat (1) cracks (1) in (1) the (1) sun (1) =5? Actually "Bat cracks in the sun": Bat(1) cracks(1) in(1) the(1) sun(1) =5. Good. Line2: Ball(1) flies(1) o-ver(2) gree

In [20]:
result["messages"][-1].usage_metadata

{'input_tokens': 340, 'output_tokens': 1024, 'total_tokens': 1364}

In [21]:
result["messages"][-1].response_metadata

{'token_usage': {'prompt_tokens': 340,
  'completion_tokens': 1024,
  'total_tokens': 1364,
  'prompt_tokens_details': {'cached_tokens': 0}},
 'model_name': '@cf/nvidia/nemotron-3-120b-a12b'}

### Try it on your own!
Change the system prompt, use the `pretty_printer` to print some messages or dig through `results` on your own. Notice the Human, AI and Tool messages and some of their associated metadata. Notice how the final results provide a complete history of the agents activity!

In [23]:
agent = create_agent(
    model=lc_model,
    tools=[check_haiku_lines],
    system_prompt="You are a poet who talks about geopolotics. You always write haikus.",
)

In [24]:
result = agent.invoke({"messages": "Please write me a poem"})

for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

checking haiku, it has 3 lines:
 Silent missiles wait
Diplomacy frays and snaps
Peace hangs by a thread
================================ Human Message =================================

Please write me a poem
================================== Ai Message ==================================

[{'type': 'thinking', 'thinking': 'The user wants a poem. I am a poet who talks about geopolitics and always writes haikus. So I need to write a haiku about geopolitics. I should ensure it\'s a haiku: 5-7-5 syllable structure, 3 lines. I need to check with the tool to verify it\'s correct. I\'ll first compose a haiku about geopolitics, then use the check_haiku_lines tool to verify it has exactly 3 lines. The tool checks if the given haiku text has exactly 3 lines, not syllable count. But the user expects a haiku, which traditionally is 5-7-5 syllables. However the tool only checks line count. I should still aim for 5-7-5. I\'ll write a haiku, then check line count. If it\'s correct, I\'ll output it. 